# Línea base temporal para `price_per_sqm`

La línea base se ajusta con **OLS de solo intercepto**: al no incluir características del inmueble, su predicción es la media de `price_per_sqm` observada **solo en entrenamiento**. No necesita resolver todavía la colinealidad de las columnas one-hot.

El dataset final no contiene `date_posted`, por lo que recuperamos la fecha del archivo transformado anterior. Verificamos que las filas mantengan el mismo orden antes de separar entrenamiento (hasta el 17/01/2025) y prueba (desde el 18/01/2025). Conservamos todos los registros para medir el desempeño inicial.

In [15]:
# Importamos Path para ubicar los datasets desde la carpeta Parcial.
from pathlib import Path

# Importamos pandas para leer las tablas y calcular la línea base.
import pandas as pd

# Importamos statsmodels para ajustar OLS explícitamente.
import statsmodels.api as sm

# Cargamos las fechas del dataset anterior sin modificar el archivo final.
RUTA_CON_FECHA = Path("data_sets_transformados/secondary_market_date_transformado.csv")
df_con_fecha = pd.read_csv(RUTA_CON_FECHA, parse_dates=["date_posted"])

# Cargamos los precios por metro cuadrado del dataset preparado.
RUTA_FINAL = Path("data_sets_transformados/secondary_market_modelo_final.csv")
df_modelo_final = pd.read_csv(RUTA_FINAL)

# Verificamos longitud y columnas originales compartidas para asegurar la alineación fila por fila.
columnas_control = ["price_per_sqm", "total_area", "building_age"]
if len(df_con_fecha) != len(df_modelo_final) or not df_con_fecha[columnas_control].equals(df_modelo_final[columnas_control]):
    raise ValueError("Los datasets no están alineados; no se puede recuperar la fecha por posición.")

# Verificamos que todas las publicaciones tengan una fecha válida.
if df_con_fecha["date_posted"].isna().any():
    raise ValueError("Hay fechas de publicación inválidas; revisarlas antes de dividir los datos.")

# Definimos el mismo corte cronológico que usaremos para comparar los futuros modelos.
fecha_corte = pd.Timestamp("2025-01-17")
es_entrenamiento = df_con_fecha["date_posted"] <= fecha_corte

# Separamos el objetivo en entrenamiento y prueba sin usar los predictores.
y_entrenamiento = df_modelo_final.loc[es_entrenamiento, "price_per_sqm"]
y_prueba = df_modelo_final.loc[~es_entrenamiento, "price_per_sqm"]

# Comprobamos que ambos períodos contengan registros antes de calcular métricas.
if y_entrenamiento.empty or y_prueba.empty:
    raise ValueError("El corte temporal dejó vacío entrenamiento o prueba.")

# La OLS de solo intercepto aprende la media de entrenamiento.
diseno_base_entrenamiento = pd.DataFrame({"const": 1.0}, index=y_entrenamiento.index)
modelo_base = sm.OLS(y_entrenamiento, diseno_base_entrenamiento).fit()
prediccion_constante = float(modelo_base.params["const"])

# Predecimos el intercepto estimado para cada publicación de prueba.
diseno_base_prueba = pd.DataFrame({"const": 1.0}, index=y_prueba.index)
predicciones_baseline = modelo_base.predict(diseno_base_prueba).rename("prediccion_price_per_sqm")

# Calculamos los errores entre el valor observado y el predicho.
errores_baseline = y_prueba - predicciones_baseline
mae_baseline = errores_baseline.abs().mean()
rmse_baseline = errores_baseline.pow(2).mean() ** 0.5

# Calculamos R² usando la variabilidad del conjunto de prueba como referencia.
variabilidad_prueba = (y_prueba - y_prueba.mean()).pow(2).sum()
r2_baseline = 1 - errores_baseline.pow(2).sum() / variabilidad_prueba

# Mostramos las métricas para poder compararlas con la regresión lineal posterior.
print(f"Entrenamiento: {len(y_entrenamiento)} | Prueba: {len(y_prueba)}")
print(f"Predicción constante: {prediccion_constante:,.2f} RUB/m²")
print(f"MAE: {mae_baseline:,.2f} RUB/m² | RMSE: {rmse_baseline:,.2f} RUB/m² | R²: {r2_baseline:.6f}")

# Comparamos algunas publicaciones de prueba con las predicciones de la línea base.
comparacion_baseline = pd.DataFrame({
    "date_posted": df_con_fecha.loc[y_prueba.index, "date_posted"],
    "price_per_sqm_real": y_prueba,
    "price_per_sqm_predicho": predicciones_baseline,
    "error_absoluto": errores_baseline.abs()
})
comparacion_muestra = comparacion_baseline.head().copy()
comparacion_muestra[["price_per_sqm_predicho", "error_absoluto"]] = comparacion_muestra[["price_per_sqm_predicho", "error_absoluto"]].round(2)
display(comparacion_muestra)

Entrenamiento: 40011 | Prueba: 9989
Predicción constante: 190,627.91 RUB/m²
MAE: 84,777.96 RUB/m² | RMSE: 123,010.33 RUB/m² | R²: -0.043143


,date_posted,price_per_sqm_real,price_per_sqm_predicho,error_absoluto
1,2025-08-13,247300,190627.91,56672.09
3,2025-04-08,282400,190627.91,91772.09
6,2026-01-31,201000,190627.91,10372.09
18,2025-04-10,167200,190627.91,23427.91
20,2025-12-02,234600,190627.91,43972.09


### Registro opcional de la línea base en MLflow

La celda anterior **calcula y muestra todo sin MLflow**. Ejecutá esta celda solo si querés registrar métricas y predicciones en el servidor local indicado en el README.

In [16]:
# Importamos MLflow solo en la celda que registra la línea base.
import mlflow

# Apuntamos al servidor local e identificamos el experimento de modelos.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - Modelos")

# Registramos el intercepto OLS, el corte temporal y las mismas métricas de prueba.
with mlflow.start_run(run_name="baseline_ols_intercepto"):
    mlflow.log_params({
        "modelo": "OLS_solo_intercepto",
        "objetivo": "price_per_sqm",
        "fecha_corte": str(fecha_corte.date()),
        "n_entrenamiento": len(y_entrenamiento),
        "n_prueba": len(y_prueba)
    })
    mlflow.log_metrics({
        "intercepto_ols": prediccion_constante,
        "mae_test": float(mae_baseline),
        "rmse_test": float(rmse_baseline),
        "r2_test": float(r2_baseline)
    })
    # Guardamos la predicción y el error de cada publicación evaluada.
    mlflow.log_table(
        comparacion_baseline.reset_index().rename(columns={"index": "fila_original"}).assign(
            date_posted=lambda tabla: tabla["date_posted"].dt.strftime("%Y-%m-%d")
        ),
        artifact_file="predicciones/prueba_baseline.json"
    )

🏃 View run baseline_ols_intercepto at: http://127.0.0.1:5000/#/experiments/1/runs/64bba5bfbb4c4440a9b586f740d78c0d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Interpretación de la línea base

Con 40.011 publicaciones de entrenamiento y 9.989 de prueba, la predicción constante es **190.627,91 RUB/m²**. En prueba, el MAE es **84.777,96 RUB/m²**, el RMSE **123.010,33 RUB/m²** y el R² **-0,043143**. El R² negativo indica que predecir la media del período de entrenamiento fue algo peor que usar la media del propio período de prueba; no significa que el cálculo falló.

A continuación ajustamos un OLS múltiple sobre el **mismo corte** y comparamos sus métricas.

## Regresión OLS múltiple sobre el mismo corte temporal

Para comparar con la línea base usamos exactamente las mismas filas de entrenamiento y prueba. Ajustamos **statsmodels OLS con intercepto explícito**; `price_per_sqm` queda fuera de los predictores. Como las variables categóricas ya están codificadas con one-hot completo, quitamos una categoría de referencia por grupo para evitar la dependencia lineal exacta con el intercepto. `has_balcony_binary` ya es una sola columna binaria.

In [17]:
# statsmodels se importó en la celda de cálculo de la línea base.
# Definimos los grupos one-hot que necesitan una categoría de referencia.
prefijos_categoricos = [
    "district_", "building_type_", "seller_type_",
    "renovation_", "metro_station_", "metro_distance_type_"
]

# Quitamos una categoría de cada grupo para evitar colinealidad exacta con el intercepto.
columnas_referencia = [
    next(columna for columna in df_modelo_final.columns if columna.startswith(prefijo))
    for prefijo in prefijos_categoricos
]

# Separamos las características X del valor real que queremos predecir.
X = df_modelo_final.drop(columns=["price_per_sqm", *columnas_referencia])

# Comprobamos que las características sean numéricas y no tengan faltantes.
if X.select_dtypes(exclude="number").shape[1] or X.isna().any().any():
    raise ValueError("Los predictores deben ser numéricos y no tener valores faltantes.")

# Usamos las mismas filas cronológicas que se utilizaron para la línea base.
X_entrenamiento = X.loc[es_entrenamiento]
X_prueba = X.loc[~es_entrenamiento]

# Agregamos el intercepto explícito y usamos floats para el ajuste OLS.
X_entrenamiento_ols = sm.add_constant(X_entrenamiento.astype("float64"), has_constant="add")
X_prueba_ols = sm.add_constant(X_prueba.astype("float64"), has_constant="add")

# Ajustamos OLS únicamente con las publicaciones y precios de entrenamiento.
modelo_lineal = sm.OLS(y_entrenamiento, X_entrenamiento_ols).fit()

# Predecimos los precios por metro cuadrado de las mismas publicaciones de prueba.
predicciones_lineales = pd.Series(
    modelo_lineal.predict(X_prueba_ols), index=y_prueba.index, name="prediccion_lineal"
)

# Calculamos los errores para evaluar el modelo con las mismas métricas de la línea base.
errores_lineales = y_prueba - predicciones_lineales
mae_lineal = errores_lineales.abs().mean()
rmse_lineal = errores_lineales.pow(2).mean() ** 0.5
r2_lineal = 1 - errores_lineales.pow(2).sum() / variabilidad_prueba

# Presentamos juntas las métricas de ambos modelos para compararlos.
comparacion_modelos = pd.DataFrame({
    "modelo": ["Línea base (media)", "Regresión lineal"],
    "MAE (RUB/m²)": [mae_baseline, mae_lineal],
    "RMSE (RUB/m²)": [rmse_baseline, rmse_lineal],
    "R²": [r2_baseline, r2_lineal]
})
print(f"Predictores: {X.shape[1]} | Entrenamiento: {len(X_entrenamiento)} | Prueba: {len(X_prueba)}")
print(f"Categorías de referencia excluidas: {columnas_referencia}")
display(comparacion_modelos.round(3))

# Mostramos predicciones de ambos modelos para las mismas filas de prueba.
ejemplos_modelos = pd.DataFrame({
    "precio_real": y_prueba,
    "linea_base": predicciones_baseline,
    "regresion_lineal": predicciones_lineales
})
display(ejemplos_modelos.head().round(2))

# El resumen de ajuste describe entrenamiento; MAE, RMSE y R² anteriores son de prueba.
print(modelo_lineal.summary().tables[0])

Predictores: 257 | Entrenamiento: 40011 | Prueba: 9989
Categorías de referencia excluidas: ['district_Aeroport', 'building_type_brick', 'seller_type_agency', 'renovation_cosmetic', 'metro_station_Aleksandrovsky Sad', 'metro_distance_type_transport']


,modelo,MAE (RUB/m²),RMSE (RUB/m²),R²
0,Línea base (media),84777.963,123010.328,-0.043
1,Regresión lineal,36467.237,44622.513,0.863


,precio_real,linea_base,regresion_lineal
1,247300,190627.91,271501.55
3,282400,190627.91,279928.23
6,201000,190627.91,300212.05
18,167200,190627.91,175103.06
20,234600,190627.91,263063.53


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                     1528.
Date:                Sat, 10 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:26:41   Log-Likelihood:            -4.7379e+05
No. Observations:               40011   AIC:                         9.481e+05
Df Residuals:                   39753   BIC:                         9.503e+05
Df Model:                         257                                         
Covariance Type:            nonrobust                                         


### Registro opcional de la regresión OLS en MLflow

Esta celda registra el modelo OLS, sus métricas de prueba y la tabla de coeficientes en el experimento de modelos.

In [ ]:
import mlflow

# Apuntamos al servidor local e identificamos el experimento de modelos.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - Modelos")

# Registramos parámetros, métricas y tablas del OLS evaluado sobre prueba temporal.
with mlflow.start_run(run_name="regresion_ols_temporal"):
    mlflow.log_params({
        "modelo": "statsmodels_OLS",
        "objetivo": "price_per_sqm",
        "fecha_corte": str(fecha_corte.date()),
        "n_entrenamiento": len(y_entrenamiento),
        "n_prueba": len(y_prueba),
        "n_predictores": X.shape[1],
        "categorias_referencia": ", ".join(columnas_referencia)
    })
    mlflow.log_metrics({
        "mae_test": float(mae_lineal),
        "rmse_test": float(rmse_lineal),
        "r2_test": float(r2_lineal),
        "r2_train": float(modelo_lineal.rsquared),
        "aic_train": float(modelo_lineal.aic),
        "bic_train": float(modelo_lineal.bic)
    })
    mlflow.log_table(comparacion_modelos, artifact_file="comparacion/modelos.json")
    mlflow.log_table(modelo_lineal.params.rename("coeficiente").reset_index(), artifact_file="modelo/coeficientes.json")

### Interpretación de la regresión OLS

El OLS múltiple mejora claramente a la línea base: reduce el MAE de **84.777,96** a **36.467,24 RUB/m²**, el RMSE de **123.010,33** a **44.622,51 RUB/m²** y alcanza un R² de **0,862732** en las mismas 9.989 publicaciones de prueba. Esto indica que las características del inmueble aportan información relevante para describir `price_per_sqm`.

## Distribuciones de variables en MLflow

Esta sección registra estadísticas descriptivas y distribuciones de las variables del entrenamiento. No modifica los datos ni elimina observaciones.

In [ ]:
import matplotlib.pyplot as plt

# Usamos únicamente entrenamiento para describir las variables del modelo.
df_auditoria = df_modelo_final.loc[es_entrenamiento].copy()
resumen_variables = df_auditoria.describe().T.reset_index().rename(columns={"index": "variable"})

# Resumimos las columnas binarias y categóricas ya codificadas.
columnas_dummy = [columna for columna in df_auditoria if columna.startswith(tuple(prefijos_categoricos))]
frecuencias_dummy = pd.DataFrame({
    "variable": columnas_dummy,
    "cantidad": df_auditoria[columnas_dummy].sum().to_numpy(),
    "proporcion": df_auditoria[columnas_dummy].mean().to_numpy()
})

display(resumen_variables.round(4))
display(frecuencias_dummy.round(4))

### Registro de distribuciones en MLflow

La siguiente celda guarda las tablas y gráficos descriptivos del entrenamiento como artefactos.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - Auditoría")

with mlflow.start_run(run_name="distribuciones_entrenamiento"):
    mlflow.log_params({
        "periodo": "entrenamiento",
        "fecha_corte": str(fecha_corte.date()),
        "n_registros": len(df_auditoria),
        "n_variables": df_auditoria.shape[1]
    })
    mlflow.log_table(resumen_variables, artifact_file="tablas/resumen_variables.json")
    mlflow.log_table(frecuencias_dummy, artifact_file="tablas/frecuencias_one_hot.json")

    for variable in df_auditoria.select_dtypes(include="number").columns:
        figura, eje = plt.subplots(figsize=(8, 4))
        eje.hist(df_auditoria[variable], bins=40)
        eje.set(title=f"Distribución de {variable}", xlabel=variable, ylabel="Registros")
        figura.tight_layout()
        mlflow.log_figure(figura, f"distribuciones/{variable}.png")
        plt.close(figura)

## Análisis de coeficientes del OLS

Analizamos los coeficientes estimados con sus `t-values`, `p-values` e intervalos de confianza al 95%. También indicamos el signo y la magnitud absoluta del efecto.

In [18]:
# Obtenemos los intervalos de confianza al 95% de cada coeficiente OLS.
intervalos_confianza = modelo_lineal.conf_int(alpha=0.05)

# Construimos una tabla con coeficientes, significancia, intervalo y magnitud.
tabla_coeficientes_ols = pd.DataFrame({
    "variable": modelo_lineal.params.index,
    "coeficiente": modelo_lineal.params.to_numpy(),
    "t_value": modelo_lineal.tvalues.to_numpy(),
    "p_value": modelo_lineal.pvalues.to_numpy(),
    "ic_inferior_95": intervalos_confianza[0].to_numpy(),
    "ic_superior_95": intervalos_confianza[1].to_numpy()
})

# Clasificamos el sentido del efecto y calculamos su magnitud absoluta.
tabla_coeficientes_ols["signo"] = tabla_coeficientes_ols["coeficiente"].map(
    lambda valor: "positivo" if valor > 0 else "negativo" if valor < 0 else "neutro"
)
tabla_coeficientes_ols["magnitud_absoluta"] = tabla_coeficientes_ols["coeficiente"].abs()

# Ordenamos por magnitud para identificar los efectos estimados más grandes.
tabla_coeficientes_ordenada = tabla_coeficientes_ols.sort_values("magnitud_absoluta", ascending=False)

# Mostramos la tabla completa para consultar cualquier variable y sus estadísticas.
display(tabla_coeficientes_ordenada.round(6))

,variable,coeficiente,t_value,p_value,ic_inferior_95,ic_superior_95,signo,magnitud_absoluta
0,const,-3.102136e+07,-32.454566,0.000000,-3.289483e+07,-2.914790e+07,negativo,3.102136e+07
47,district_Khamovniki,2.913101e+05,96.429058,0.000000,2.853889e+05,2.972312e+05,positivo,2.913101e+05
123,district_Tverskoy,2.691800e+05,91.096976,0.000000,2.633884e+05,2.749716e+05,positivo,2.691800e+05
19,district_Arbat,2.677543e+05,88.342324,0.000000,2.618138e+05,2.736949e+05,positivo,2.677543e+05
96,district_Presnensky,2.453501e+05,82.486043,0.000000,2.395201e+05,2.511801e+05,positivo,2.453501e+05
...,...,...,...,...,...,...,...,...
246,metro_station_Ugreshskaya,1.089662e+02,0.039398,0.968573,-5.311980e+03,5.529913e+03,positivo,1.089662e+02
8,total_floors,3.922579e+01,1.227742,0.219551,-2.339607e+01,1.018477e+02,positivo,3.922579e+01
4,living_area,2.872191e+01,0.508228,0.611296,-8.204645e+01,1.394903e+02,positivo,2.872191e+01
15,building_age,9.175644e+00,0.537759,0.590746,-2.426772e+01,4.261901e+01,positivo,9.175644e+00


### Interpretación de los coeficientes

Un coeficiente positivo indica un aumento estimado de `price_per_sqm` cuando la variable aumenta, manteniendo constantes las demás. Un coeficiente negativo indica una reducción. En las variables one-hot, el efecto se interpreta respecto de la categoría de referencia excluida.

El `t-value` compara el coeficiente con su error estándar; el `p-value` evalúa la hipótesis de que el coeficiente sea cero; y el intervalo de confianza muestra un rango plausible para el efecto.